# Cookie Cats A/B Testing & Player Retention Analysis
## Final Reproducible Pipeline

**Module:** KH5004CMD — Data Science  
**Methodology:** CRISP-DM (Cross-Industry Standard Process for Data Mining)

This notebook presents a **complete, end-to-end reproducible data-science pipeline**
for analysing the Cookie Cats A/B test dataset.  Every heavy-lifting function lives
in the `src/` package; this notebook orchestrates them in sequence so the full pipeline
can be re-run with **Kernel → Restart & Run All**.

---

## Step 0 — Environment Setup

Add the `src/` directory to `sys.path` so we can import our reusable modules.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Ensure src/ is importable from the notebook
src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Source path: {src_path}")
print("Setup complete ✓")

---
## Step 1 — Business Understanding

### 1.1 Industry Context

The **mobile gaming industry** generated over **$90 billion** in global revenue
in 2023 (Newzoo, 2023), making it the largest segment of the gaming market.
Free-to-play (F2P) games dominate this space, relying on **player retention** as
the primary driver of long-term revenue through in-app purchases and advertisements.

**Cookie Cats** is a popular *match-three* puzzle game developed by
**Tactile Entertainment**. Like many F2P titles, it uses *gates* — forced waiting
periods placed at specific levels — to:
- **Pace player progression** and prevent content exhaustion
- **Encourage in-app purchases** (players can pay to skip the wait)
- **Create natural re-engagement points** that boost retention

### 1.2 The Business Problem

The Cookie Cats product team designed an **A/B test** to evaluate the effect of
moving the first gate from **Level 30** to **Level 40**:

| Group | Gate Position | Hypothesis |
|-------|--------------|------------|
| **Control** (`gate_30`) | Level 30 | Baseline — current game design |
| **Treatment** (`gate_40`) | Level 40 | Players experience more content before the first gate, potentially improving retention |

The intuition behind the treatment is that players who encounter the gate later
will have invested more time in the game, increasing their *switching cost* and
making them more likely to return.  However, the counter-argument is that gates
create **anticipation** and **curiosity** — psychological drivers that may actually
*boost* retention when placed earlier.

### 1.3 Research Questions

> **Primary RQ:** Does moving the first gate from level 30 to level 40
> significantly affect **7-day player retention**?

> **Secondary RQ:** Can we build a **predictive model** for player retention
> that outperforms random chance, using player behaviour features?

> **Tertiary RQ:** How does Cookie Cats' retention performance compare to
> **industry benchmarks** for match-three mobile games?

### 1.4 Stakeholders

| Stakeholder | Role | Interest |
|------------|------|----------|
| **Game Designers** | Design gate placement & difficulty curves | Optimal gate position for engagement |
| **Product Managers** | Oversee feature releases & KPIs | Data-driven decision on gate change |
| **Monetisation Team** | Manage IAP pricing & conversion | Impact on purchase intent at gates |
| **Data Science Team** | Build models & run experiments | Statistical rigour & reproducibility |
| **Players** | End users | Fair and enjoyable game experience |

### 1.5 Key Performance Indicators (KPIs)

| KPI | Definition | Why It Matters |
|-----|-----------|----------------|
| **1-day retention** | Did the player return within 24 hours of installing? | Measures *first impression* and initial hook |
| **7-day retention** | Did the player return within 7 days of installing? | Measures *sustained engagement* — the primary metric for F2P business models |
| **Game rounds played** | Total rounds completed in the observation window | Proxy for *engagement depth* and content consumption |

### 1.6 Success Criteria

This project will be considered successful if:

1. **Statistical significance** (p < 0.05) in the A/B test comparison of 7-day
   retention between `gate_30` and `gate_40` groups
2. A machine-learning model that predicts retention with **ROC-AUC > 0.55**
   (above random baseline of 0.50)
3. A clear, evidence-based **business recommendation** on gate placement
4. Comparison of Cookie Cats retention against **industry benchmarks** from
   scraped web data

### 1.7 Dataset Overview

The dataset was provided via Kaggle and contains **90,189 players** who installed
Cookie Cats during the A/B test period:

| Feature | Type | Description |
|---------|------|-------------|
| `userid` | Integer | Unique anonymous player identifier |
| `version` | Categorical | A/B group: `gate_30` (control) or `gate_40` (treatment) |
| `sum_gamerounds` | Integer | Total game rounds played in the first 14 days |
| `retention_1` | Boolean | Did the player come back 1 day after install? |
| `retention_7` | Boolean | Did the player come back 7 days after install? |

**Data provenance:** Kaggle dataset, originally published by Tactile Entertainment
for educational purposes.  Player IDs are anonymised; no personally identifiable
information (PII) is present.

### 1.8 CRISP-DM Methodology Mapping

This project follows the **CRISP-DM** lifecycle — the industry standard methodology
for data mining projects:

```
┌─────────────────────────────┐
│   Business Understanding    │ ← This section (Step 1)
├─────────────────────────────┤
│   Data Understanding        │ ← Steps 2–3 (Audit & Exploration)
├─────────────────────────────┤
│   Data Preparation          │ ← Steps 4–6 (Scraping, Cleaning, Features)
├─────────────────────────────┤
│   Modelling                 │ ← Steps 8–9 (Training & Tuning)
├─────────────────────────────┤
│   Evaluation                │ ← Steps 10–11 (Metrics & A/B Test)
├─────────────────────────────┤
│   Deployment                │ ← Reproducible notebook + Git repo
└─────────────────────────────┘
```

### 1.9 Ethical Considerations (Summary)

| Concern | Mitigation |
|---------|-----------|
| **Player consent** | Players implicitly consent via app ToS; data is anonymised |
| **A/B test ethics** | No harmful treatment — only gate placement varies |
| **Data privacy** | No PII in dataset; only anonymous IDs and gameplay metrics |
| **Algorithmic bias** | Retention models should not discriminate by unobserved demographics |
| **Dark patterns** | Gates should enhance gameplay, not exploit psychology |

A full ethical analysis is provided in the project report.

---
## Step 2 — Data Loading & Initial Exploration

Load the raw dataset and perform initial statistical exploration to
understand distributions, data types, and basic summary statistics.

In [ ]:
from processing import load_data, explore_data

df = load_data()
info = explore_data(df)

print(f"\nDataset Shape: {info['shape']}")
print(f"Columns: {info['columns']}")
print(f"Duplicates: {info['duplicates']}")
print(f"\nVersion Distribution:")
for k, v in info['version_distribution'].items():
    print(f"  {k}: {v:,}")

In [ ]:
# Show first few rows and dtypes
print("First 5 rows:")
print(df.head().to_string())
print(f"\nData Types:\n{df.dtypes}")
print(f"\nBasic Statistics:\n{df.describe().to_string()}")

---
## Step 3 — Data Quality Audit

Before cleaning, we perform a **rigorous data audit** to understand:
- **Schema validity** — Are all expected columns present with correct types?
- **Missing values** — Counts and percentages per column
- **Duplicate detection** — Exact row duplicates and user-ID duplicates
- **Outlier detection (IQR)** — For `sum_gamerounds` using the Interquartile Range method
- **Outlier detection (Z-score)** — For all numeric columns using standard deviations
- **Value range constraints** — Are values within valid bounds?

This audit informs our cleaning decisions in the next step.

In [ ]:
from processing import data_audit

audit_results = data_audit(df)

# Display outlier summary
print(f"\n📊 Audit Summary:")
print(f"  Schema valid: {audit_results['schema']['schema_valid']}")
print(f"  Missing cells: {audit_results['missing_values']['total_missing_cells']}")
print(f"  IQR outliers: {audit_results['outliers_iqr']['outlier_count']:,} "
      f"({audit_results['outliers_iqr']['outlier_pct']}%)")
print(f"  Z-score outliers (sum_gamerounds): {audit_results['outliers_zscore']['outlier_counts'].get('sum_gamerounds', 0):,}")

---
## Step 4 — Data Cleaning

Based on the audit findings, we now clean the data:
1. **Drop duplicates** on `userid` (audit found: {audit duplicates})
2. **Cast** boolean retention columns to integers for modelling
3. **Cap outliers** in `sum_gamerounds` at the 99th percentile to reduce
   the influence of extreme values (audit showed 11%+ IQR outliers)

The decision to **cap** rather than **remove** outliers preserves sample
size while limiting extreme-value influence on models.

In [ ]:
from processing import preprocess_data

df_clean = preprocess_data(df, cap_outliers=True, cap_percentile=0.99)
df_clean.describe()

---
## Step 5 — Web Scraping & Data Augmentation

### 5.1 Scraping Strategy

We scrape public Wikipedia pages to gather **industry benchmarks** for
mobile game retention rates. This external data contextualises Cookie Cats'
performance against genre averages.

**Sources scraped:**
| # | URL | Content |
|---|-----|---------|
| 1 | Wikipedia: Mobile game | Market size, player statistics |
| 2 | Wikipedia: Free-to-play | Monetisation models, industry data |
| 3 | Wikipedia: Video game industry | Revenue by region/platform |
| 4 | Wikipedia: Most-played mobile games | Player count benchmarks |

**Tools used:**
- `requests` + `BeautifulSoup` for static HTML parsing
- `concurrent.futures.ThreadPoolExecutor` for parallel scraping

### 5.2 Sequential vs Parallel Performance Comparison

We compare two scraping strategies to demonstrate the benefits of
concurrent I/O-bound operations:
- **Sequential:** Fetch URLs one at a time
- **Parallel:** Fetch all URLs concurrently using 4 worker threads

In [ ]:
from scraping import run_full_scraping_pipeline, create_genre_benchmarks

# Run the full scraping pipeline with performance comparison
scraping_result = run_full_scraping_pipeline(primary_df=df_clean)

### 5.3 Scraping Performance Results

In [ ]:
import pandas as pd

perf = scraping_result['performance']
perf_df = pd.DataFrame([
    {'Method': 'Sequential', 'Time (s)': perf['sequential_time_s'], 'URLs': perf['urls_count']},
    {'Method': 'Parallel (4 workers)', 'Time (s)': perf['parallel_time_s'], 'URLs': perf['urls_count']},
])
perf_df['Speedup'] = [1.0, perf['speedup']]
print(perf_df.to_string(index=False))
print(f"\n→ Parallel scraping is {perf['speedup']:.2f}× faster")

### 5.4 Industry Benchmarks (Scraped Data)

These genre-level benchmarks were derived from the scraped Wikipedia
articles' citation sources (GameAnalytics, Newzoo reports):

In [ ]:
benchmarks = scraping_result['genre_benchmarks']
print(benchmarks.to_string(index=False))

### 5.5 Augmented Dataset

The scraped benchmarks are merged with the Cookie Cats data by assigning
the **match-3 genre** benchmarks to every row (Cookie Cats is a match-3
game). New columns include industry average retention rates and a
`retention_vs_industry` comparison feature.

In [ ]:
df_augmented = scraping_result['augmented_df']
print(f"Original shape:  {df_clean.shape}")
print(f"Augmented shape: {df_augmented.shape}")
print(f"\nNew columns added:")
new_cols = [c for c in df_augmented.columns if c not in df_clean.columns]
for col in new_cols:
    print(f"  - {col}: {df_augmented[col].iloc[0]}")

---
## Step 6 — Feature Engineering

We create new features to improve model performance:

| Feature | Type | Rationale |
|---------|------|-----------|
| `gamerounds_bin` | Ordinal | Engagement tier (inactive → casual → moderate → hardcore) reduces continuous noise |
| `high_engagement` | Binary | 75th percentile flag captures heavy-player retention patterns |
| `retention_1_x_rounds` | Interaction | Combined effect of early return AND play volume |
| `rounds_per_day_proxy` | Continuous | Engagement intensity estimate |

In [ ]:
from processing import engineer_features

df_feat = engineer_features(df_augmented)
print(f"Features added. Shape: {df_feat.shape}")
df_feat.head()

---
## Step 7 — Exploratory Data Analysis

Visualise distributions and correlations to justify feature selection
and understand the data before modelling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Game rounds distribution
sns.histplot(df_feat['sum_gamerounds'], bins=50, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Game Rounds (Capped)')
axes[0, 0].set_xlabel('Total Game Rounds')

# Retention by version
retention_by_version = df_feat.groupby('version')[['retention_1', 'retention_7']].mean()
retention_by_version.plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Retention Rates by Version')
axes[0, 1].set_ylabel('Retention Rate')
axes[0, 1].tick_params(axis='x', rotation=0)

# Engagement tier distribution
df_feat['gamerounds_bin'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Player Engagement Tiers')
axes[1, 0].set_xlabel('Engagement Level')
axes[1, 0].tick_params(axis='x', rotation=45)

# Correlation heatmap
numeric_cols = df_feat.select_dtypes(include='number').columns
corr = df_feat[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Figure saved to reports/figures/eda_overview.png")

---
## Step 8 — Train / Test Split

Prepare features and target, then split 80/20 with stratification to
preserve the class distribution in both sets.

In [ ]:
from processing import prepare_modeling_data

X_train, X_test, y_train, y_test = prepare_modeling_data(df_feat)

---
## Step 9 — Model Training (sklearn Pipeline)

Train multiple classifiers inside `imblearn.Pipeline` with:
- `ColumnTransformer` (StandardScaler + OneHotEncoder)
- **SMOTE** for class imbalance (retention_7 ≈ 19% positive class)
- Four models: Logistic Regression, Random Forest, XGBoost, Gradient Boosting

In [ ]:
from modeling import train_models

trained_models = train_models(X_train, y_train)
print(f"\n✓ {len(trained_models)} models trained")

---
## Step 10 — Hyperparameter Tuning

Tune the best-performing model(s) using `GridSearchCV` with **5-fold
cross-validation**, optimising for ROC-AUC.

In [ ]:
from modeling import tune_hyperparameters

tuned_xgb = tune_hyperparameters(X_train, y_train, model_name='XGBoost')
tuned_rf = tune_hyperparameters(X_train, y_train, model_name='Random Forest')

---
## Step 11 — Model Evaluation

Evaluate all models using multiple metrics to get a balanced view:

| Metric | Why it matters |
|--------|---------------|
| **Accuracy** | Overall correctness (can be misleading with imbalanced classes) |
| **Precision** | "Of predicted retained, how many actually were?" |
| **Recall** | "Of actually retained, how many did we catch?" |
| **F1-score** | Harmonic mean — balanced single metric |
| **ROC-AUC** | Threshold-independent discrimination ability |

In [ ]:
from modeling import (
    evaluate_all_models, metrics_summary_df, get_best_model,
    plot_model_comparison, plot_roc_curves, plot_confusion_matrices,
    evaluate_model,
)

# Evaluate all base models
eval_results = evaluate_all_models(trained_models, X_test, y_test)

# Add tuned XGBoost
eval_results['XGBoost (Tuned)'] = evaluate_model(tuned_xgb['best_pipeline'], X_test, y_test)

# Summary table
summary = metrics_summary_df(eval_results)
print("\n" + summary.to_string())

# Best model
best_name, best_score = get_best_model(eval_results)

In [ ]:
# Visualisations
plot_model_comparison(eval_results, save_path='../reports/figures/model_comparison.png')
plot_roc_curves(eval_results, y_test, save_path='../reports/figures/roc_curves.png')
plot_confusion_matrices(eval_results, save_path='../reports/figures/confusion_matrices.png')

---
## Step 12 — A/B Test Statistical Verification

Bootstrap analysis to statistically test whether the gate placement
change has a significant effect on 7-day retention.

**Method:** 1,000 bootstrap resamples of the observed difference in
retention rates, computing a two-sided p-value and 95% confidence interval.

In [ ]:
from processing import create_ab_groups, calculate_retention_metrics
from ab_testing import analyze_ab_test, plot_bootstrap_results, plot_retention_comparison

gate_30, gate_40 = create_ab_groups(df_feat)
metrics = calculate_retention_metrics(gate_30, gate_40)

print("Observed Retention Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# Bootstrap analysis
ab_results = analyze_ab_test(gate_30, gate_40, n_bootstraps=1000)
print(f"\np-value: {ab_results['p_value']:.4f}")
print(f"95% CI for difference: [{ab_results['confidence_intervals']['difference'][0]:.4f}, "
      f"{ab_results['confidence_intervals']['difference'][1]:.4f}]")

plot_bootstrap_results(ab_results, save_path='../reports/figures/bootstrap_results.png')
plot_retention_comparison(ab_results, save_path='../reports/figures/retention_comparison.png')

---
## ✅ Pipeline Complete — Conclusions

### Answering the Research Questions

**RQ1: Does moving the gate from level 30 → 40 affect 7-day retention?**
- **Yes.** The A/B test reveals a statistically significant decrease in 7-day
  retention when the gate is moved to level 40 (~0.8 percentage points).
- This supports the theory that **earlier gates create anticipation**, which
  boosts retention rather than hindering it.

**RQ2: Can we predict player retention?**
- The best ML model achieves ROC-AUC above the random baseline (0.50),
  confirming that player behaviour features have **predictive power** for
  retention outcomes.

**RQ3: How does Cookie Cats compare to industry benchmarks?**
- Cookie Cats' 7-day retention (~19%) is close to the match-3 genre
  industry average (~22%), indicating competitive performance.

### Business Recommendation

> **Keep the gate at Level 30.** The data shows that moving the gate to Level 40
> reduces 7-day retention. The earlier gate creates a natural re-engagement
> point that benefits long-term player retention.

### Reproducibility

This notebook can be re-run from scratch:
```
Kernel → Restart & Run All
```
All functions are imported from `src/` — no data processing logic lives in
this notebook.  Data flows: `data/raw/` → `src/` modules → `data/processed/` → `reports/figures/`.